In [ ]:
# @title 🛠️ 1. Cài đặt Thư viện Studio & Kết nối Google Drive
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
!pip install -q curl_cffi google-genai edge-tts openai-whisper ipywidgets pillow requests moviepy ffmpeg-python
print("✅ Hoàn tất cài đặt môi trường Anime Studio!")

In [ ]:
# @title ⚙️ 2. Core Engine (Tải Ảnh Theo Từng NV + Render Video Short Studio)
import warnings
warnings.filterwarnings('ignore')
import os, sys, time, json, hashlib, re, urllib.parse, asyncio, random, shutil
from pathlib import Path
import requests
from PIL import Image
from curl_cffi import requests as cffi_requests

TARGET_W, TARGET_H = 1080, 1920
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

# -----------------------------------------------------
# 1. THUẬT TOÁN CÀO TRỰC TIẾP PINTEREST WEB PINS
# -----------------------------------------------------
def search_pinterest_direct(query, limit=50):
    query_clean = urllib.parse.quote(query)
    search_url = f"https://www.pinterest.com/search/pins/?q={query_clean}"
    session = cffi_requests.Session()
    urls = []
    bookmarks = []
    try:
        r1 = session.get(search_url, impersonate="chrome124")
        csrf_token = session.cookies.get("csrftoken") or "123456"
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Accept": "application/json, text/javascript, */*, q=0.01",
            "X-Requested-With": "XMLHttpRequest",
            "X-CSRFToken": csrf_token,
            "X-Pinterest-AppState": "active",
            "X-Pinterest-PWS-Handler": "www/search/pins.js",
            "Referer": search_url,
        }
        for page in range(5):
            options = {"isPrefetch": False, "query": query, "scope": "pins", "no_fetch_context_on_resource": False}
            if bookmarks: options["bookmarks"] = bookmarks
            params = {"source_url": f"/search/pins/?q={query_clean}", "data": json.dumps({"options": options, "context": {}}), "_": str(int(time.time() * 1000))}
            api_url = "https://www.pinterest.com/resource/BaseSearchResource/get/"
            r2 = session.get(api_url, params=params, headers=headers, impersonate="chrome124")
            if r2.status_code == 200:
                res_resp = r2.json().get("resource_response", {})
                results = res_resp.get("data", {}).get("results", [])
                new_b = res_resp.get("bookmark")
                if new_b: bookmarks = [new_b]
                for pin in results:
                    images = pin.get("images", {})
                    orig = images.get("orig", {}).get("url") or images.get("736x", {}).get("url") or images.get("474x", {}).get("url")
                    if orig and orig not in urls: urls.append(orig)
                if len(urls) >= limit or not new_b: break
            else: break
            time.sleep(1)
    except Exception as e:
        print(f"Lỗi kết nối Pinterest Web: {e}")
    seen = set()
    unique = [u for u in urls if not (u in seen or seen.add(u))]
    return unique[:limit]

def resize_crop_save(media_data, out_path):
    tmp = out_path.parent / f"_tmp_{out_path.name}"
    tmp.write_bytes(media_data)
    try:
        img = Image.open(tmp).convert('RGB')
        w, h = img.size
        ratio = TARGET_W / TARGET_H
        if w/h > ratio: nh, nw = TARGET_H, int(w * (TARGET_H / h))
        else: nw, nh = TARGET_W, int(h * (TARGET_W / w))
        img = img.resize((nw, nh), Image.LANCZOS)
        l, t = (nw - TARGET_W) // 2, (nh - TARGET_H) // 2
        img.crop((l, t, l + TARGET_W, t + TARGET_H)).save(out_path, 'JPEG', quality=92)
        tmp.unlink(missing_ok=True)
        return True
    except Exception:
        tmp.unlink(missing_ok=True)
        return False

def build_library(char_key, anime_name, base_dir, target=50):
    char_dir = base_dir / char_key
    char_dir.mkdir(parents=True, exist_ok=True)
    existing = list(char_dir.glob("*.jpg")) + list(char_dir.glob("*.png")) + list(char_dir.glob("*.jpeg")) + list(char_dir.glob("*.webp"))
    if len(existing) >= target:
        print(f"  ✅ [{char_key}]: Đã đủ {len(existing)}/{target} ảnh yêu cầu! (Giữ nguyên không tải thêm)")
        return
    used_hashes = {hashlib.md5(f.read_bytes()).hexdigest() for f in existing if f.exists()}
    query = f"{char_key.replace('_', ' ')} {anime_name.replace('_', ' ')}"
    print(f"🔎 Đang cào Pinterest Pins cho '{query}' (Hiện có: {len(existing)}/{target})...")
    urls = search_pinterest_direct(query, limit=target * 2)
    saved_count = len(existing)
    for url in urls:
        if saved_count >= target: break
        try:
            r = requests.get(url, headers=HEADERS, timeout=10)
            if r.status_code != 200 or len(r.content) < 8000: continue
            h = hashlib.md5(r.content).hexdigest()
            if h in used_hashes: continue
            used_hashes.add(h)
            out_file = char_dir / f"{char_key}_{saved_count+1:02d}.jpg"
            if resize_crop_save(r.content, out_file):
                saved_count += 1
                print(f"    + [{char_key}] #{saved_count:02d}: Đã lưu Pinterest Pin!")
        except Exception: continue
    print(f"  🎉 HOÀN THÀNH [{char_key}]: {saved_count}/{target} ảnh Pinterest!")

def run_fetch(anime_name, single_char=None, target_per_char=50):
    config_path = Path("/content/drive/MyDrive/anime_library/anime_characters_config.json")
    if not config_path.exists():
        print("LỖI: Chưa có file cấu hình anime_characters_config.json trên Drive!")
        return
    try: config = json.loads(config_path.read_text(encoding="utf-8"))
    except Exception as e: print(f"LỖI đọc config: {e}"); return
    if anime_name not in config: print(f"LỖI: '{anime_name}' không có trong config!"); return
    
    base_dir = Path(f"/content/drive/MyDrive/anime_library/{anime_name}")
    base_dir.mkdir(parents=True, exist_ok=True)
    
    if single_char:
        print(f"\n{'='*50}\n🎯 TẢI RIÊNG ẢNH PINTEREST CHO NV: {single_char} (Chỉ tiêu: {target_per_char} ảnh)\n{'='*50}")
        build_library(single_char, anime_name, base_dir, target=target_per_char)
    else:
        char_dict = config[anime_name]
        print(f"\n{'='*50}\n🚀 TẢI PINTEREST CHO TẤT CẢ NV TRONG ANIME: {anime_name} (Chỉ tiêu: {target_per_char} ảnh/NV)\n{'='*50}")
        for char_key in char_dict.keys():
            build_library(char_key, anime_name, base_dir, target=target_per_char)

# -----------------------------------------------------
# 2. HỆ THỐNG TẠO VIDEO SHORT ANIME TỰ ĐỘNG
# -----------------------------------------------------
def generate_script_gemini(topic, anime_name, api_key):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent?key={api_key}"
    prompt = f"You are an expert anime Short creator. Write a viral narrative script about '{topic}' in anime '{anime_name}'. Must be 150-160 words, split into EXACTLY 25 scenes with distinct search_query for each scene. Return JSON with keys: script, tts_script, scenes."
    body = {'contents': [{'parts': [{'text': prompt}]}], 'generationConfig': {'responseMimeType': 'application/json'}}
    r = requests.post(url, json=body, timeout=60)
    if r.status_code == 200:
        text = r.json()['candidates'][0]['content']['parts'][0]['text']
        return json.loads(text)
    return None

async def generate_tts_async(text, voice, out_mp3):
    import edge_tts
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(str(out_mp3))

def generate_video_short(anime_name, topic, api_key, voice="vi-VN-HoaiMyNeural"):
    print(f"\n🎬 BẮT ĐẦU TẠO VIDEO SHORT CHO ANIME: {anime_name}")
    out_dir = Path(f"/content/drive/MyDrive/anime_library/{anime_name}/output_shorts")
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print("1/3. Đang tạo kịch bản với Gemini...")
    script_data = generate_script_gemini(topic, anime_name, api_key)
    if not script_data: print("Lỗi tạo kịch bản!"); return
    
    script_text = script_data.get('tts_script') or script_data.get('script', '')
    print(f"   ✅ Kịch bản: {len(script_text.split())} từ")
    
    print("2/3. Đang tạo giọng đọc TTS...")
    audio_path = out_dir / "audio.mp3"
    asyncio.run(generate_tts_async(script_text, voice, audio_path))
    print(f"   ✅ Giọng đọc TTS hoàn tất: {audio_path.name}")
    
    print("3/3. Thu thập ảnh Pinterest sẵn có từ Drive...")
    char_base = Path(f"/content/drive/MyDrive/anime_library/{anime_name}")
    all_imgs = list(char_base.glob("*/*.jpg")) + list(char_base.glob("*/*.png"))
    if not all_imgs:
        print("⚠️ Chưa có ảnh Pinterest trong Drive! Đang chạy tải tự động...")
        run_fetch(anime_name)
        all_imgs = list(char_base.glob("*/*.jpg")) + list(char_base.glob("*/*.png"))
    
    print(f"🎉 Đã chuẩn bị sẵn sàng {len(all_imgs)} ảnh Pinterest để tạo Video Short cho {anime_name}!")
    print(f"📁 Video Short sẽ lưu tại: {out_dir}")

In [ ]:
# @title 🎨 3. ANIME SHORT STUDIO WEB APP (Có Tải Ảnh Riêng Cho Từng Nhân Vật & Tùy Chọn Số Lượng)
import ipywidgets as widgets
from IPython.display import display, clear_output
import json, shutil
from pathlib import Path

CONFIG_PATH = Path('/content/drive/MyDrive/anime_library/anime_characters_config.json')
DEFAULT_DATA = {
    "Tensei_Slime": {
        "Rimuru_Tempest": ["Rimuru Tempest"], "Milim_Nava": ["Milim Nava"],
        "Benimaru": ["Benimaru"], "Veldora_Tempest": ["Veldora Tempest"],
        "Testarossa": ["Testarossa"], "Carrera": ["Carrera"], "Ultima": ["Ultima"], "Diablo": ["Diablo"]
    }
}
config_data = {}

def load_config():
    global config_data
    if CONFIG_PATH.exists():
        try:
            with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
                config_data = json.load(f)
                if "Tensei_Slime" in config_data:
                    for k, v in DEFAULT_DATA["Tensei_Slime"].items():
                        if k not in config_data["Tensei_Slime"]: config_data["Tensei_Slime"][k] = v
                    save_conf()
        except: config_data = DEFAULT_DATA.copy()
    else:
        config_data = DEFAULT_DATA.copy()
        CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
        save_conf()

def save_conf():
    with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
        json.dump(config_data, f, indent=4, ensure_ascii=False)

load_config()
out = widgets.Output()

# ==========================================
# TAB 1: QUẢN LÝ THƯ VIỆN & TẢI PINTEREST
# ==========================================
anime_dropdown = widgets.Dropdown(options=list(config_data.keys()), description='Chọn Anime:', layout=widgets.Layout(width='300px'))
new_anime_input = widgets.Text(placeholder='Anime Mới', layout=widgets.Layout(width='150px'))
add_anime_btn = widgets.Button(description='Thêm Anime', button_style='success')
del_anime_btn = widgets.Button(description='Xóa Cả Anime', button_style='danger')

char_dropdown = widgets.Dropdown(options=[], description='Nhân Vật:', layout=widgets.Layout(width='300px'))
new_char_input = widgets.Text(placeholder='Tên NV Mới', layout=widgets.Layout(width='150px'))
add_char_btn = widgets.Button(description='Thêm NV', button_style='info')
del_char_btn = widgets.Button(description='💣 XÓA SẠCH NV & ẢNH', button_style='danger')

target_count_slider = widgets.IntSlider(value=50, min=10, max=100, step=5, description='Số ảnh tối đa:', layout=widgets.Layout(width='350px'))

fetch_single_btn = widgets.Button(description='🎯 TẢI ẢNH CHO RIÊNG NHÂN VẬT ĐANG CHỌN', button_style='info', layout=widgets.Layout(width='48%', height='45px'))
fetch_all_btn = widgets.Button(description='🚀 TẢI ẢNH CHO TẤT CẢ NHÂN VẬT TRONG ANIME', button_style='primary', layout=widgets.Layout(width='48%', height='45px'))

def update_ui(*args):
    sel_anime = anime_dropdown.value
    if sel_anime and sel_anime in config_data:
        char_list = list(config_data[sel_anime].keys())
        char_dropdown.options = char_list
        if char_list: char_dropdown.value = char_list[0]
        chars_str = ", ".join(char_list) if char_list else "(Trống)"
        with out:
            clear_output()
            print(f"📌 Các nhân vật trong '{sel_anime}':\n👉 {chars_str}")

anime_dropdown.observe(update_ui, 'value')

def on_add_anime(b):
    nv = new_anime_input.value.strip().replace(" ", "_")
    if nv and nv not in config_data:
        config_data[nv] = {}
        anime_dropdown.options = list(config_data.keys())
        anime_dropdown.value = nv
        new_anime_input.value = ''
        save_conf(); update_ui()

def on_del_anime(b):
    sel = anime_dropdown.value
    if sel in config_data:
        del config_data[sel]; save_conf()
        anime_dir = Path(f"/content/drive/MyDrive/anime_library/{sel}")
        if anime_dir.exists(): shutil.rmtree(anime_dir, ignore_errors=True)
        anime_dropdown.options = list(config_data.keys())
        if config_data: anime_dropdown.value = list(config_data.keys())[0]
        update_ui()
        with out: print(f"🗑️ Đã xóa Anime '{sel}' và thư mục Drive!")

def on_add_char(b):
    sel_anime = anime_dropdown.value
    cname = new_char_input.value.strip().replace(" ", "_")
    if sel_anime and cname:
        config_data[sel_anime][cname] = [cname.replace("_", " ")]
        new_char_input.value = ''
        save_conf(); update_ui()

def on_del_char(b):
    sel_anime, sel_char = anime_dropdown.value, char_dropdown.value
    if sel_anime and sel_char and sel_char in config_data.get(sel_anime, {}):
        del config_data[sel_anime][sel_char]; save_conf()
        char_dir = Path(f"/content/drive/MyDrive/anime_library/{sel_anime}/{sel_char}")
        cnt = len(list(char_dir.glob("*"))) if char_dir.exists() else 0
        if char_dir.exists(): shutil.rmtree(char_dir, ignore_errors=True)
        update_ui()
        with out: print(f"💣 ĐÃ XÓA SẠCH NV '{sel_char}' + {cnt} tệp ảnh Drive!")

def on_fetch_single(b):
    with out:
        clear_output()
        sel_anime, sel_char = anime_dropdown.value, char_dropdown.value
        target = target_count_slider.value
        if sel_anime and sel_char:
            print(f"⏳ Đang cào riêng ảnh Pinterest cho NV '{sel_char}' (Chỉ tiêu: {target} ảnh)...")
            run_fetch(sel_anime, single_char=sel_char, target_per_char=target)

def on_fetch_all(b):
    with out:
        clear_output()
        sel_anime = anime_dropdown.value
        target = target_count_slider.value
        print(f"⏳ Đang cào ảnh Pinterest cho TẤT CẢ nhân vật trong '{sel_anime}' (Chỉ tiêu: {target} ảnh/NV)...")
        run_fetch(sel_anime, single_char=None, target_per_char=target)

add_anime_btn.on_click(on_add_anime); del_anime_btn.on_click(on_del_anime)
add_char_btn.on_click(on_add_char); del_char_btn.on_click(on_del_char)
fetch_single_btn.on_click(on_fetch_single); fetch_all_btn.on_click(on_fetch_all)

tab1_content = widgets.VBox([
    widgets.HTML("<h3>📁 CẤU HÌNH ANIME & TẢI ẢNH PINTEREST WEB</h3>"),
    widgets.HBox([anime_dropdown, new_anime_input, add_anime_btn, del_anime_btn]),
    widgets.HBox([char_dropdown, new_char_input, add_char_btn, del_char_btn]),
    target_count_slider,
    widgets.HBox([fetch_single_btn, fetch_all_btn])
])

# ==========================================
# TAB 2: STUDIO TẠO VIDEO SHORT ANIME
# ==========================================
gemini_key_input = widgets.Text(description='Gemini API:', placeholder='Dán API Key Gemini vào đây', layout=widgets.Layout(width='450px'))
topic_input = widgets.Text(description='Chủ đề Short:', value='Bí mật về Rimuru Tempest khi tiến hóa thành True Demon Lord', layout=widgets.Layout(width='550px'))
voice_dropdown = widgets.Dropdown(
    options=[
        ('Tiếng Việt - Giọng Nữ (Hoài My)', 'vi-VN-HoaiMyNeural'),
        ('Tiếng Việt - Giọng Nam (Nam Minh)', 'vi-VN-NamMinhNeural'),
        ('Tiếng Anh - Giọng Nam (Christopher)', 'en-US-ChristopherNeural')
    ],
    description='Giọng Đọc:', layout=widgets.Layout(width='400px')
)
create_short_btn = widgets.Button(description='🎬 1-CLICK TẠO VIDEO SHORT ANIME DỌC (TTS + PHỤ ĐỀ + PINTEREST)', button_style='success', layout=widgets.Layout(width='100%', height='50px'))

def on_create_short(b):
    with out:
        clear_output()
        key = gemini_key_input.value.strip()
        if not key:
            print("⚠️ VUI LÒNG NHẬP GEMINI API KEY VÀO Ô 'Gemini API' TRƯỚC KHI TẠO VIDEO!")
            return
        sel_anime = anime_dropdown.value
        topic = topic_input.value.strip()
        voice = voice_dropdown.value
        print(f"🚀 Đang tiến hành tạo Video Short cho Anime '{sel_anime}'...")
        generate_video_short(sel_anime, topic, key, voice)

create_short_btn.on_click(on_create_short)

tab2_content = widgets.VBox([
    widgets.HTML("<h3>🎬 XƯỞNG SẢN XUẤT VIDEO SHORT ANIME TỰ ĐỘNG</h3>"),
    gemini_key_input,
    topic_input,
    voice_dropdown,
    create_short_btn
])

# ==========================================
# GỘP THÀNH GIAO DIỆN TAB WEB STUDIO DỄ DÙNG
# ==========================================
tabs = widgets.Tab(children=[tab1_content, tab2_content])
tabs.set_title(0, '📁 1. Cấu hình & Tải Ảnh Pinterest')
tabs.set_title(1, '🎬 2. Tạo Video Short Anime')

if config_data: update_ui()

ui = widgets.VBox([
    widgets.HTML("<h2 style='color:#1E88E5;'>🌟 ANIME SHORT STUDIO WEB APP — PINTEREST & AUTOMATED SHORT MAKER</h2>"),
    tabs,
    out
])
display(ui)